In [40]:
import pandas as pd
import numpy as np

# Load all files
results      = pd.read_csv('../data/raw/results.csv')
drivers      = pd.read_csv('../data/raw/drivers.csv')
constructors = pd.read_csv('../data/raw/constructors.csv')
races        = pd.read_csv('../data/raw/races.csv')
qualifying   = pd.read_csv('../data/raw/qualifying.csv')
pit_stops    = pd.read_csv('../data/raw/pit_stops.csv')
standings    = pd.read_csv('../data/raw/driver_standings.csv')

# Replace all '\N' (missing marker in this dataset) with NaN
for df in [results, drivers, constructors, races, qualifying, pit_stops, standings]:
    df.replace('\\N', np.nan, inplace=True)

print("✅ Loaded & cleaned null markers!")
weather = pd.read_csv('../data/processed/weather.csv')
print(f"✅ Weather data loaded: {weather.shape}")

✅ Loaded & cleaned null markers!
✅ Weather data loaded: (797, 5)


In [41]:
# Start with results (one row = one driver in one race)
df = results[['raceId', 'driverId', 'constructorId', 'grid', 'positionOrder',
              'points', 'statusId']].copy()

# Attach race info (year, round, circuitId)
df = df.merge(races[['raceId', 'year', 'round', 'circuitId']], on='raceId', how='left')

# Attach driver info (nationality)
df = df.merge(drivers[['driverId', 'nationality']], on='driverId', how='left')

# Attach constructor info (constructor nationality)
df = df.merge(
    constructors[['constructorId', 'nationality']].rename(
        columns={'nationality': 'constructor_nationality'}
    ),
    on='constructorId', how='left'
)

df['grid']          = pd.to_numeric(df['grid'], errors='coerce')
df['positionOrder'] = pd.to_numeric(df['positionOrder'], errors='coerce')
df['points']        = pd.to_numeric(df['points'], errors='coerce')

print(f"✅ Base dataframe shape: {df.shape}")
df.head()
# Merge weather data
df = df.merge(weather[['raceId', 'is_wet_race', 'precipitation_mm']], 
              on='raceId', how='left')

# Fill missing weather (races before 1980) as dry
df['is_wet_race']       = df['is_wet_race'].fillna(0).astype(int)
df['precipitation_mm']  = df['precipitation_mm'].fillna(0.0)

print(f"✅ Weather merged! Wet races in dataset: {df['is_wet_race'].sum()}")

✅ Base dataframe shape: (26759, 12)
✅ Weather merged! Wet races in dataset: 6181


In [42]:
qualifying['position'] = pd.to_numeric(qualifying['position'], errors='coerce')

quali_clean = qualifying[['raceId', 'driverId', 'position']].rename(
    columns={'position': 'quali_position'}
)

df = df.merge(quali_clean, on=['raceId', 'driverId'], how='left')

print("✅ Qualifying position added!")

✅ Qualifying position added!


In [43]:
# Sort by driver and race
df = df.sort_values(['driverId', 'year', 'round']).reset_index(drop=True)

# Rolling average finishing position over last 5 races (per driver)
df['driver_rolling_avg_finish'] = (
    df.groupby('driverId')['positionOrder']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

print("✅ Driver recent form added!")

✅ Driver recent form added!


In [44]:
# Average finishing position of constructor over last 5 races
df['constructor_rolling_avg_finish'] = (
    df.groupby('constructorId')['positionOrder']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

print("✅ Constructor recent form added!")

✅ Constructor recent form added!


In [45]:
df = df.sort_values(['driverId', 'circuitId', 'year']).reset_index(drop=True)

df['driver_circuit_avg_finish'] = (
    df.groupby(['driverId', 'circuitId'])['positionOrder']
    .transform(lambda x: x.shift(1).expanding().mean())
)

print("✅ Driver circuit history added!")

✅ Driver circuit history added!


In [46]:
standings['position'] = pd.to_numeric(standings['position'], errors='coerce')
standings['points']   = pd.to_numeric(standings['points'], errors='coerce')

# Get standing BEFORE this race (shift within season per driver)
standings_sorted = standings.sort_values(['driverId', 'raceId'])
standings_sorted['prev_standing'] = (
    standings_sorted.groupby('driverId')['position'].shift(1)
)

df = df.merge(
    standings_sorted[['raceId', 'driverId', 'prev_standing']],
    on=['raceId', 'driverId'],
    how='left'
)

print("✅ Championship standings added!")

✅ Championship standings added!


In [47]:
pit_counts = pit_stops.groupby(['raceId', 'driverId']).size().reset_index(name='pit_stop_count')

df = df.merge(pit_counts, on=['raceId', 'driverId'], how='left')
df['pit_stop_count'] = df['pit_stop_count'].fillna(0)

print("✅ Pit stop counts added!")

✅ Pit stop counts added!


In [48]:
feature_cols = [
    'grid',
    'quali_position',
    'driver_rolling_avg_finish',
    'constructor_rolling_avg_finish',
    'driver_circuit_avg_finish',
    'prev_standing',
    'pit_stop_count',
    'is_wet_race',           # ← new
    'precipitation_mm',      # ← new
    'year',
    'round',
    'driverId',
    'constructorId',
    'circuitId',
    'positionOrder'
]

df_model = df[feature_cols].copy()
df_model = df_model.dropna(subset=['positionOrder', 'grid', 'driver_rolling_avg_finish'])

print(f"✅ Final dataset shape: {df_model.shape}")
print(f"\n🌧️  Wet races in model: {df_model['is_wet_race'].sum()}")
print(f"☀️   Dry races in model: {(df_model['is_wet_race']==0).sum()}")
df_model.head()

✅ Final dataset shape: (25898, 15)

🌧️  Wet races in model: 6110
☀️   Dry races in model: 19788


,grid,quali_position,driver_rolling_avg_finish,constructor_rolling_avg_finish,driver_circuit_avg_finish,prev_standing,pit_stop_count,is_wet_race,precipitation_mm,year,round,driverId,constructorId,circuitId,positionOrder
1,1,1.0,6.6,6.6,3.0,5.0,0.0,0,0.0,2008,1,1,1,1,1
2,18,15.0,5.6,5.6,2.0,NaN,0.0,0,0.0,2009,1,1,1,1,20
3,11,11.0,5.8,5.8,8.0,3.0,0.0,1,3.7,2010,2,1,1,1,6
4,2,2.0,6.4,6.4,7.5,4.0,2.0,0,0.0,2011,1,1,1,1,2
5,1,1.0,7.4,7.4,6.4,5.0,2.0,0,0.0,2012,1,1,1,1,3


In [49]:
df_model.to_csv('../data/processed/f1_features.csv', index=False)
print("✅ Saved to data/processed/f1_features.csv")

✅ Saved to data/processed/f1_features.csv
